# Notebook 01: YOLOv11 Quickstart

**Goal**: Run detection on a single image and video using pre-trained YOLOv11, understand outputs, then fine-tune on KITTI.

## What you'll learn
1. Load and run YOLOv11 on images/video
2. Understand detection outputs (boxes, scores, class ids)
3. Key differences between YOLO11n / m / x and RT-DETR
4. Fine-tune on KITTI vehicle detection
5. Export to ONNX for deployment

In [ ]:
# Install if needed
# !pip install ultralytics>=8.3.0 opencv-python matplotlib

## Part 1: Run pre-trained YOLOv11 (zero-shot on COCO)

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Load pre-trained YOLOv11 medium (auto-downloads ~25 MB)
model = YOLO('yolo11m.pt')
print(f'Parameters: {sum(p.numel() for p in model.model.parameters()):,}')
print(f'Classes: {len(model.names)}')

In [ ]:
# Run on a sample image (change path to your image)
# Using a built-in test image from ultralytics
results = model.predict(
    'https://ultralytics.com/images/bus.jpg',
    conf=0.25,
    iou=0.45,
    verbose=False,
)

# Visualize
r = results[0]
annotated = r.plot()                    # annotated image as numpy array
plt.figure(figsize=(12, 8))
plt.imshow(annotated[..., ::-1])        # BGR → RGB
plt.axis('off')
plt.title(f'Detected {len(r.boxes)} objects')
plt.show()

In [ ]:
# Inspect raw outputs
print('Detection outputs:')
print(f'  Boxes (xyxy): {r.boxes.xyxy.shape}')      # Nx4
print(f'  Scores:       {r.boxes.conf.shape}')       # N
print(f'  Class IDs:    {r.boxes.cls.shape}')        # N

print('\nTop-5 detections:')
for i in range(min(5, len(r.boxes))):
    box   = r.boxes.xyxy[i].tolist()
    score = float(r.boxes.conf[i])
    cls   = int(r.boxes.cls[i])
    name  = r.names[cls]
    print(f'  {name:15s} score={score:.3f}  box=[{box[0]:.0f},{box[1]:.0f},{box[2]:.0f},{box[3]:.0f}]')

## Part 2: Model Comparison (YOLO11n vs m vs x vs RT-DETR)

| Model | Params | mAP@50:95 | Speed (T4) | Best for |
|-------|--------|-----------|------------|----------|
| yolo11n | 2.6M | 39.5 | 1.5ms | Edge/mobile |
| yolo11m | 20.1M | 51.5 | 4.7ms | Balance |
| yolo11x | 56.9M | 54.7 | 11.3ms | Max accuracy |
| rtdetr-l | 32.9M | 53.0 | 9.2ms | Crowded scenes |

RT-DETR advantage: no NMS step, better for crowded/overlapping vehicles.

In [ ]:
import time

def benchmark_model(model_name: str, image_path: str, n_runs: int = 50):
    model = YOLO(model_name)
    # Warmup
    for _ in range(3):
        model.predict(image_path, verbose=False)
    # Benchmark
    t0 = time.perf_counter()
    for _ in range(n_runs):
        r = model.predict(image_path, verbose=False)
    elapsed = time.perf_counter() - t0
    fps = n_runs / elapsed
    n_det = len(r[0].boxes)
    print(f'{model_name:15s}  {fps:6.1f} FPS  {n_det} detections')

IMAGE = 'https://ultralytics.com/images/bus.jpg'
for name in ['yolo11n.pt', 'yolo11m.pt', 'rtdetr-l.pt']:
    benchmark_model(name, IMAGE)

## Part 3: Fine-tune on KITTI

**Prerequisites:**
1. Download KITTI: `bash scripts/download_kitti.sh`
2. Convert to YOLO: `python data/prepare_kitti.py --data_root data/kitti --convert_yolo`

In [ ]:
# Fine-tune YOLOv11m on KITTI
# This uses transfer learning: start from COCO weights, adapt to KITTI

model = YOLO('yolo11m.pt')

results = model.train(
    data='data/kitti_yolo/kitti.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,            # GPU 0
    amp=True,            # FP16 for 2x speedup
    workers=8,
    name='kitti_finetune',
    # Transfer learning: freeze first 10 layers, train rest
    # freeze=10,
)

print(f'Best mAP@50:    {results.results_dict["metrics/mAP50(B)"]:.4f}')
print(f'Best mAP@50:95: {results.results_dict["metrics/mAP50-95(B)"]:.4f}')

## Part 4: Multi-Object Tracking with ByteTrack

In [ ]:
# ByteTrack is built into Ultralytics — just add .track()
model = YOLO('runs/detect/kitti_finetune/weights/best.pt')

# Track on a video (replace with your video path)
# results = model.track(
#     source='your_driving_video.mp4',
#     tracker='bytetrack.yaml',
#     conf=0.25,
#     persist=True,            # keep track IDs across frames
#     save=True,               # save annotated video
# )

# Inspect track IDs
# for r in results:
#     if r.boxes.id is not None:
#         track_ids = r.boxes.id.int().tolist()
#         class_names = [r.names[int(c)] for c in r.boxes.cls]
#         print(f'Tracks: {list(zip(track_ids, class_names))}')

print('Track example ready — uncomment and provide a video path')

## Part 5: Export to ONNX

In [ ]:
model = YOLO('runs/detect/kitti_finetune/weights/best.pt')

# Export to ONNX (works on any hardware)
path = model.export(format='onnx', imgsz=640, simplify=True)
print(f'ONNX model: {path}')

# Export to TensorRT (NVIDIA only, 3-5x faster inference)
# path = model.export(format='engine', imgsz=640, half=True)
# print(f'TensorRT engine: {path}')